# Агент суммаризации JSON
MCP + LangChain + LangGraph + Groq. CPU достаточно. Ключ хранится только в Colab Secrets или вводится скрыто через getpass.

Этот ноутбук запускает настоящий агентный граф, а не цикл суммаризации статей. Подробности каждой функции — в README.

In [1]:
from pathlib import Path
print("Existing results:", list(Path("/content/agents-homework/results_reproduction").glob("*")))
if "completed" in globals():
    print(completed.stdout)
    print(completed.stderr)

Existing results: []


In [2]:
import os, subprocess, sys
from pathlib import Path
repo = Path("/content/agents-homework")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/evelinashakhnazaryan/agents-homework.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed")

Dependencies installed


## Проверки без API
Включают настоящий MCP-сервер и LangGraph. Решения модели в инфраструктурном тесте заменены тестовыми сообщениями; это не реальный прогон Groq.

In [3]:
subprocess.run([sys.executable, "-m", "unittest", "-v", "test_agent"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', '-v', 'test_agent'], returncode=0)

## Подключение Groq
Добавьте секрет `GROQ_API_KEY` и разрешите доступ этому ноутбуку. Если секрета нет, появится скрытое поле ввода. Значение не выводится и не сохраняется в ноутбуке.

In [4]:
from google.colab import userdata
from getpass import getpass
try:
    api_key = userdata.get("GROQ_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    api_key = getpass("Groq API key: ")
if not api_key:
    raise ValueError("Нужен ключ Groq")
os.environ["GROQ_API_KEY"] = api_key
os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"
del api_key
print("Groq key configured (value hidden)")

Groq key configured (value hidden)


## Запуск агента
Модель самостоятельно вызывает чтение → суммаризацию по ID → сохранение. Новый запуск использует уникальную папку. При исчерпании квоты запуск завершится ошибкой и оставит трассу.

In [5]:
from datetime import datetime, timezone
from uuid import uuid4
run_dir = Path("results_reproduction") / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6])
run_dir.mkdir(parents=True)
output = run_dir / "summaries.json"
with (run_dir / "run.log").open("w") as log:
    proc = subprocess.Popen([sys.executable, "-u", "agent.py", "--input", "articles.json", "--output", str(output)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = proc.wait()
print("Exit code:", returncode)
assert returncode == 0, "See run.log and trace for details"

[09/25/26 19:19:47] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
Tools: read_articles
[09/25/26 19:19:48] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
Tools: summarize_article
[09/25/26 19:19:49] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
                    INFO     HTTP Request: POST                  _client.py:1740
                             https://api.groq.com/openai/v1/chat                
                             /completions "HTTP/1.1 200 OK"                     
Tools: summarize_article
[09/25/26 19:19:50] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
                    INFO     HTTP Requ

AssertionError: See run.log and trace for details

## Проверка и просмотр результатов
Проверяем хеш входа, все ID, число записей и непустые резюме. Затем выводим резюме для ручной оценки.

In [12]:
import json
from agent import verify_output
result = verify_output(Path("articles.json"), output)
trace = json.loads(output.with_suffix(".trace.json").read_text(encoding="utf-8"))
assert trace["success"]
print("Articles:", result["count"], "| Model:", result["model"])
for article in result["articles"]:
    print("\n", article["id"], article.get("title", ""))
    print(article["summary"])
calls = [call["name"] for event in trace["events"] for call in event.get("tool_calls", [])]
print("\nTool calls:", calls)

Articles: 10 | Model: openai/gpt-oss-120b

 1 Регламент оформления командировок и возмещения расходов
С 1 января 2026 года в компании введён новый регламент командировок: заявки подаются в системе TravelHub не позднее 5 рабочих дней до выезда и согласуются руководителем подразделения и финансовым контролером. Базовый лимит на проживание составляет 9 000 ₽ за ночь по России, 12 000 ₽ по СНГ и 160 € по Европе; суточные – 2 500 ₽ для Specialist и Senior и 3 500 ₽ для руководителей групп. Авиабилеты покупаются только в эконом‑классе, кроме случаев перелётов более 8 часов при наличии медицинского заключения. Все закрывающие документы должны быть загружены в течение 7 календарных дней после возвращения; при несоблюдении срока расходы переводятся в статус личных и удерживаются из ближайшей выплаты.

 2 Отчет по инциденту ИБ-24-117: фишинговая рассылка
14 марта 2026 г. в 09:12 SOC зафиксировал всплеск переходов по подозрительным ссылкам из корпоративной почты; к 09:25 инцидент получил уровень 

## Скачать результат
Архив содержит реальные ответы и трассу; ключа в нём нет. Выполненный ноутбук можно скачать через меню «Файл → Скачать → Скачать IPYNB».

In [6]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(run_dir), "zip", run_dir)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import json, base64
from IPython.display import HTML, display
trace = json.loads(output.with_suffix(".trace.json").read_text())
print(json.dumps(trace["events"][-3:], ensure_ascii=False, indent=2))
data = base64.b64encode(Path(archive).read_bytes()).decode()
display(HTML(f'<a download="failed_run.zip" href="data:application/zip;base64,{data}">Download failed run archive</a>'))

[
  {
    "node": "agent",
    "type": "ai",
    "content": "",
    "tool_calls": [
      {
        "name": "summarize_article",
        "args": {
          "article_id": "4"
        },
        "id": "fc_7ea54484-b3ba-481d-916a-87cfdb706db1",
        "type": "tool_call"
      }
    ]
  },
  {
    "node": "tools",
    "type": "tool",
    "content": [
      {
        "type": "text",
        "text": "{\n  \"id\": \"4\",\n  \"summarized\": true,\n  \"summary_characters\": 679\n}",
        "id": "lc_aa2eacd9-7ccd-45a5-84da-46cf854fa0c9"
      }
    ],
    "tool": "summarize_article"
  },
  {
    "node": "agent",
    "type": "ai",
    "content": ""
  }
]


In [8]:
update = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True)
print(update.stdout + update.stderr)
update.check_returncode()
checked = subprocess.run([sys.executable, "-m", "unittest", "-v", "test_agent"], capture_output=True, text=True)
print(checked.stdout + checked.stderr)
checked.check_returncode()
run_dir = Path("results_reproduction") / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:6])
run_dir.mkdir(parents=True)
output = run_dir / "summaries.json"
with (run_dir / "run.log").open("w") as log:
    proc = subprocess.Popen([sys.executable, "-u", "agent.py", "--input", "articles.json", "--output", str(output)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = proc.wait()
print("Exit code:", returncode)
assert returncode == 0

Updating e97ce17..cc73a07
Fast-forward
 agent.py      | 49 ++++++++++++++++++++++++++++++++++++++++---------
 test_agent.py | 23 +++++++++++++++--------
 2 files changed, 55 insertions(+), 17 deletions(-)
From https://github.com/evelinashakhnazaryan/agents-homework
   e97ce17..cc73a07  main       -> origin/main

test_empty_summary_rejected (test_agent.ServiceTests.test_empty_summary_rejected) ... ok
test_invalid_data (test_agent.ServiceTests.test_invalid_data) ... ok
test_order_and_path_guards (test_agent.ServiceTests.test_order_and_path_guards) ... ok
test_real_mcp_and_graph_without_api (test_agent.ServiceTests.test_real_mcp_and_graph_without_api) ... [09/25/26 19:25:30] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
                    INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
               

AssertionError: 

In [9]:
os.environ["GROQ_MODEL"] = "llama-3.3-70b-versatile"
run_dir = Path("results_reproduction") / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_llama")
run_dir.mkdir(parents=True)
output = run_dir / "summaries.json"
with (run_dir / "run.log").open("w") as log:
    proc = subprocess.Popen([sys.executable, "-u", "agent.py", "--input", "articles.json", "--output", str(output)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = proc.wait()
print("Exit code:", returncode)
assert returncode == 0

[09/25/26 19:26:58] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
  + Exception Group Traceback (most recent call last):
  |   File "/content/agents-homework/agent.py", line 174, in <module>
  |     main()
  |     ~~~~^^
  |   File "/content/agents-homework/agent.py", line 170, in main
  |     asyncio.run(run(args.input, args.output, args.max_steps))
  |     ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
  |     return runner.run(main)
  |            ~~~~~~~~~~^^^^^^
  |   File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
  |     return self._loop.run_until_complete(task)
  |            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  |   File "/usr/lib/python3.13/asyncio/base_events.py", line 726, in run_until_complete
  |     return future.result()
  |            ~~~~~~~~~~~~~^^
  |   File "/content/agents-

AssertionError: 

In [10]:
from groq import Groq
print([m.id for m in Groq().models.list().data])

['allam-2-7b', 'openai/gpt-oss-20b', 'qwen/qwen3.8-27b', 'canopylabs/orpheus-arabic-saudi', 'openai/gpt-oss-120b', 'whisper-large-v3', 'openai/gpt-oss-safeguard-20b', 'meta-llama/llama-prompt-guard-2-86m', 'whisper-large-v3-turbo', 'meta-llama/llama-prompt-guard-2-22m', 'canopylabs/orpheus-v1-english']


In [11]:
os.environ["GROQ_MODEL"] = "openai/gpt-oss-120b"
run_dir = Path("results_reproduction") / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_120b")
run_dir.mkdir(parents=True)
output = run_dir / "summaries.json"
with (run_dir / "run.log").open("w") as log:
    proc = subprocess.Popen([sys.executable, "-u", "agent.py", "--input", "articles.json", "--output", str(output)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    returncode = proc.wait()
print("Exit code:", returncode)
assert returncode == 0

[09/25/26 19:29:56] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
Tools: read_articles
[09/25/26 19:29:57] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
Tools: summarize_article
[09/25/26 19:29:58] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
[09/25/26 19:29:59] INFO     HTTP Request: POST                  _client.py:1740
                             https://api.groq.com/openai/v1/chat                
                             /completions "HTTP/1.1 200 OK"                     
Tools: summarize_article
                    INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
[09/25/26 19:30:00] INFO     HTTP Requ

In [ ]:
import io, zipfile, base64, html, hashlib
from google.colab import _message
from IPython.display import HTML, display
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for p in sorted(Path("results_reproduction").rglob("*")):
        if p.is_file() and p.suffix in (".json", ".log"):
            z.write(p, str(p))
archive_bytes = buf.getvalue()
nb = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
nb["metadata"]["run_archive"] = {"format": "zip/base64", "sha256": hashlib.sha256(archive_bytes).hexdigest(), "data": base64.b64encode(archive_bytes).decode()}
export_text = json.dumps(nb, ensure_ascii=False, indent=2)
assert os.environ["GROQ_API_KEY"] not in export_text
Path("agents_executed.ipynb").write_text(export_text, encoding="utf-8")
print("Archive bytes:", len(archive_bytes), "Notebook bytes:", len(export_text.encode()))
display(HTML('<textarea aria-label="Executed notebook export" rows=5 cols=80>' + html.escape(export_text) + '</textarea>'))